In [5]:
# =====================================
# IMPORTS
# =====================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [9]:
# =====================================
# LOAD PREPROCESSED DATA
# =====================================

model_data = pd.read_pickle("../data/preprocessed_model_data.pkl")

print("Loaded preprocessed data:")
print(model_data.shape)
print(model_data.columns.tolist())


Loaded preprocessed data:
(45924, 23)
['Confirmation Year', 'Handler Region', 'Product Group', 'Product Type', 'Product Type Code', 'Industry Name', 'Industry Code', 'Service Line Code', 'Service Line Name', 'Service Detail', 'Service Program', 'Service Catalog Category', 'Service Catalog Item Number', 'Service Catalog Segment', 'Service Catalog Sub Category', 'CCN', 'Ship to Customer Region', 'Has Test Task Flag', 'Flex Standards', 'Standard Count', 'Flex Project Count', 'Test Count', 'Log_Eng_Hours']


In [10]:
# =====================================
# CREATE X AND y
# =====================================

X = model_data.drop(columns=["Log_Eng_Hours"])
y = model_data["Log_Eng_Hours"]

cat_cols = X.select_dtypes(include=["object", "str", "category"]).columns

X_encoded = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=True
)

# Clean column names for XGBoost
X_encoded.columns = (
    X_encoded.columns
    .astype(str)
    .str.replace("[", "_", regex=False)
    .str.replace("]", "_", regex=False)
    .str.replace("<", "_", regex=False)
)

print("X shape:", X_encoded.shape)
print("y shape:", y.shape)


X shape: (45924, 2615)
y shape: (45924,)


In [11]:
# =====================================
# TRAIN MODELS
# =====================================

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.001, max_iter=10000),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    )
}

results_list = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)

    mae, rmse, r2 = evaluate_model(model, X_test, y_test)

    results_list.append({
        "Model": name,
        "MAE_hours": mae,
        "RMSE_hours": rmse,
        "R2_log_scale": r2
    })

results = pd.DataFrame(results_list).sort_values("RMSE_hours")

results


Training Linear Regression...


NameError: name 'X_train' is not defined

In [ ]:
# =====================================
# FEATURE IMPORTANCE FOR BEST MODEL
# =====================================

rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(20)
